In [52]:
%pip install lightgbm


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [64]:
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid

In [54]:
train = pd.read_csv("data/prepared/train_all.csv", index_col="sample_id")
train_real = pd.read_csv("data/prepared/train_real.csv", index_col="sample_id")
val = pd.read_csv("data/prepared/val_all.csv", index_col="sample_id")
val_real = pd.read_csv("data/prepared/val_real.csv", index_col="sample_id")
test = pd.read_csv("data/prepared/test.csv", index_col="sample_id")

# Baseline - отклонение за 12 минут до прибытия

In [55]:
baseline_prediction = test["current_dev_s"]
absolute_error = (test["target_delay_s"] - baseline_prediction).abs()
base_mae = absolute_error.mean()

print(f"Baseline MAE: {base_mae:.2f} секунд")

Baseline MAE: 93.36 секунд


# Линейная регрессия с регуляризацией

In [67]:
FEATURES = ["target_stop_lon", "target_stop_lat", "time_to_target_s", "last_message_age_s", "last_speed_kmh", "mean_speed_1m_kmh", "mean_speed_3m_kmh", "mean_speed_5m_kmh", "speed_std_5m_kmh", "speed_change_5m_kmh", "stopped_share_5m", "valid_messages_5m", "distance_to_target_km", "current_dev_s", "time_sin", "time_cos"]
TARGET = "target_delay_s"

def prepare(data):
    data = data.copy()
    dt = pd.to_datetime(data["T"], errors="coerce")
    minutes = dt.dt.hour * 60 + dt.dt.minute
    data["time_sin"] = np.sin(2 * np.pi * minutes / 1440)
    data["time_cos"] = np.cos(2 * np.pi * minutes / 1440)
    return data[FEATURES]

def make_pipeline(model):
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", model)])

def tune_model(train_data, val_data, model_name):
    X_train, y_train = prepare(train_data), train_data[TARGET]
    X_val, y_val = prepare(val_data), val_data[TARGET]
    if model_name == "Ridge":
        grid = ParameterGrid({"alpha": [0.01, 0.1, 1, 10, 100, 1000]})
        make_model = lambda p: Ridge(alpha=p["alpha"])
    else:
        grid = ParameterGrid({"alpha": [0.001, 0.01, 0.1, 1, 10], "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]})
        make_model = lambda p: ElasticNet(alpha=p["alpha"], l1_ratio=p["l1_ratio"], max_iter=20000, random_state=42)
    results = []
    for params in grid:
        model = make_pipeline(make_model(params))
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        results.append({"params": params, "val_mae": mean_absolute_error(y_val, val_pred), "model": model, "name": model_name})
    return min(results, key=lambda x: x["val_mae"])

def fit_linreg_full_and_test(best_result, train_data, val_data, test_data):
    full_data = pd.concat([train_data, val_data], axis=0)
    linreg_model = clone(best_result["model"])
    linreg_model.fit(prepare(full_data), full_data[TARGET])
    test_pred = linreg_model.predict(prepare(test_data))
    return linreg_model, mean_absolute_error(test_data[TARGET], test_pred)

ridge_best_all = tune_model(train, val, "Ridge")
elasticnet_best_all = tune_model(train, val, "ElasticNet")
ridge_best_real = tune_model(train_real, val_real, "Ridge")
elasticnet_best_real = tune_model(train_real, val_real, "ElasticNet")
linreg_best_all = min([ridge_best_all, elasticnet_best_all], key=lambda x: x["val_mae"])
linreg_best_real = min([ridge_best_real, elasticnet_best_real], key=lambda x: x["val_mae"])
linreg_all_val_mae = linreg_best_all["val_mae"]
linreg_real_val_mae = linreg_best_real["val_mae"]

linreg_model_all, linreg_all_test_mae = fit_linreg_full_and_test(linreg_best_all, train, val, test)
linreg_model_real, linreg_real_test_mae = fit_linreg_full_and_test(linreg_best_real, train_real, val_real, test)

print(f"{linreg_best_all['name']} all: validation MAE = {linreg_all_val_mae:.2f} сек.; test MAE = {linreg_all_test_mae:.2f} сек.")
print(f"{linreg_best_real['name']} real: validation MAE = {linreg_real_val_mae:.2f} сек.; test MAE = {linreg_real_test_mae:.2f} сек.")

ElasticNet all: validation MAE = 104.60 сек.; test MAE = 93.08 сек.
ElasticNet real: validation MAE = 104.94 сек.; test MAE = 92.91 сек.


# CatBoost

In [57]:
CATBOOST_PARAM_GRID = ParameterGrid({
    "depth": [4, 6, 8],
    "iterations": [300, 700, 1200],
    "learning_rate": [0.03, 0.1],
    "l2_leaf_reg": [3, 10, 30]
})

def search_catboost(name, train_data, val_data):
    X_train, y_train = prepare(train_data), train_data[TARGET]
    X_val, y_val = prepare(val_data), val_data[TARGET]
    results = []

    for params in CATBOOST_PARAM_GRID:
        model = CatBoostRegressor(**params, loss_function="MAE", eval_metric="MAE", random_seed=42, verbose=False, allow_writing_files=False)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        results.append({
            "params": params,
            "train_mae": mean_absolute_error(y_train, train_pred),
            "val_mae": mean_absolute_error(y_val, val_pred)
        })

    catboost_best = min(results, key=lambda x: x["val_mae"])
    print(f"{name}: параметры = {catboost_best['params']}, MAE train = {catboost_best['train_mae']:.2f} сек., MAE validation = {catboost_best['val_mae']:.2f} сек.")
    return catboost_best

catboost_best_all = search_catboost("CatBoost на всех данных", train, val)
catboost_best_real = search_catboost("CatBoost только на реальных данных", train_real, val_real)

def fit_catboost_full_and_test(name, best_result, train_data, val_data, test_data):
    full_data = pd.concat([train_data, val_data], axis=0)
    catboost_model = CatBoostRegressor(**best_result["params"], loss_function="MAE", random_seed=42, verbose=False, allow_writing_files=False)
    catboost_model.fit(prepare(full_data), full_data[TARGET], verbose=False)
    test_pred = catboost_model.predict(prepare(test_data))
    test_mae = mean_absolute_error(test_data[TARGET], test_pred)
    print(f"{name}: MAE test = {test_mae:.2f} сек.")
    return catboost_model, test_mae

catboost_model_all, catboost_all_test_mae = fit_catboost_full_and_test("CatBoost на всех данных", catboost_best_all, train, val, test)
catboost_model_real, catboost_real_test_mae = fit_catboost_full_and_test("CatBoost только на реальных данных", catboost_best_real, train_real, val_real, test)

catboost_all_val_mae = catboost_best_all["val_mae"]
catboost_real_val_mae = catboost_best_real["val_mae"]

print(f"\nCatBoost validation: all = {catboost_all_val_mae:.2f} сек.; real = {catboost_real_val_mae:.2f} сек.")
print(f"CatBoost test: all = {catboost_all_test_mae:.2f} сек.; real = {catboost_real_test_mae:.2f} сек.")

CatBoost на всех данных: параметры = {'depth': 6, 'iterations': 300, 'l2_leaf_reg': 10, 'learning_rate': 0.1}, MAE train = 64.54 сек., MAE validation = 90.50 сек.
CatBoost только на реальных данных: параметры = {'depth': 4, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}, MAE train = 67.92 сек., MAE validation = 88.60 сек.
CatBoost на всех данных: MAE test = 48.84 сек.
CatBoost только на реальных данных: MAE test = 76.35 сек.

CatBoost validation: all = 90.50 сек.; real = 88.60 сек.
CatBoost test: all = 48.84 сек.; real = 76.35 сек.


# XGBoost

In [58]:
from xgboost import XGBRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error

XGBOOST_PARAM_GRID = ParameterGrid({
    "max_depth": [3, 4],
    "learning_rate": [0.03, 0.1],
    "reg_lambda": [10, 20]
})
XGBOOST_PARAMS = {
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
}

def search_xgboost(train_data, val_data):
    X_train, y_train = prepare(train_data), train_data[TARGET]
    X_val, y_val = prepare(val_data), val_data[TARGET]
    results = []
    for params in XGBOOST_PARAM_GRID:
        model = XGBRegressor(**XGBOOST_PARAMS, **params, n_estimators=2000, early_stopping_rounds=100)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        val_pred = model.predict(X_val)
        best_params = {**params, "n_estimators": model.best_iteration + 1}
        results.append({"params": best_params, "val_mae": mean_absolute_error(y_val, val_pred)})
    return min(results, key=lambda x: x["val_mae"])

def fit_xgboost_full_and_test(best_result, train_data, val_data, test_data):
    full_data = pd.concat([train_data, val_data], axis=0)
    xgboost_model = XGBRegressor(**XGBOOST_PARAMS, **best_result["params"])
    xgboost_model.fit(prepare(full_data), full_data[TARGET], verbose=False)
    test_pred = xgboost_model.predict(prepare(test_data))
    return xgboost_model, mean_absolute_error(test_data[TARGET], test_pred)

xgboost_best_all = search_xgboost(train, val)
xgboost_best_real = search_xgboost(train_real, val_real)
xgboost_all_val_mae = xgboost_best_all["val_mae"]
xgboost_real_val_mae = xgboost_best_real["val_mae"]

xgboost_model_all, xgboost_all_test_mae = fit_xgboost_full_and_test(xgboost_best_all, train, val, test)
xgboost_model_real, xgboost_real_test_mae = fit_xgboost_full_and_test(xgboost_best_real, train_real, val_real, test)

print(f"XGBoost all: validation MAE = {xgboost_all_val_mae:.2f} сек.; test MAE = {xgboost_all_test_mae:.2f} сек.")
print(f"XGBoost real: validation MAE = {xgboost_real_val_mae:.2f} сек.; test MAE = {xgboost_real_test_mae:.2f} сек.")

XGBoost all: validation MAE = 91.38 сек.; test MAE = 77.46 сек.
XGBoost real: validation MAE = 89.11 сек.; test MAE = 73.42 сек.


# LightGBM

In [62]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error

LIGHTGBM_PARAM_GRID = ParameterGrid({
    "num_leaves": [15, 20],
    "learning_rate": [0.03, 0.1],
    "reg_lambda": [10, 20]
})
LIGHTGBM_PARAMS = {
    "max_depth": 5,
    "min_child_samples": 30,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "objective": "regression",
    "metric": "mae",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

def search_lightgbm(train_data, val_data):
    X_train, y_train = prepare(train_data), train_data[TARGET]
    X_val, y_val = prepare(val_data), val_data[TARGET]
    results = []
    for params in LIGHTGBM_PARAM_GRID:
        model = LGBMRegressor(**LIGHTGBM_PARAMS, **params, n_estimators=2000)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
        val_pred = model.predict(X_val)
        best_params = {**params, "n_estimators": model.best_iteration_}
        results.append({"params": best_params, "val_mae": mean_absolute_error(y_val, val_pred)})
    return min(results, key=lambda x: x["val_mae"])

def fit_lightgbm_full_and_test(best_result, train_data, val_data, test_data):
    full_data = pd.concat([train_data, val_data], axis=0)
    lightgbm_model = LGBMRegressor(**LIGHTGBM_PARAMS, **best_result["params"])
    lightgbm_model.fit(prepare(full_data), full_data[TARGET])
    test_pred = lightgbm_model.predict(prepare(test_data))
    return lightgbm_model, mean_absolute_error(test_data[TARGET], test_pred)

lightgbm_best_all = search_lightgbm(train, val)
lightgbm_best_real = search_lightgbm(train_real, val_real)
lightgbm_all_val_mae = lightgbm_best_all["val_mae"]
lightgbm_real_val_mae = lightgbm_best_real["val_mae"]

lightgbm_model_all, lightgbm_all_test_mae = fit_lightgbm_full_and_test(lightgbm_best_all, train, val, test)
lightgbm_model_real, lightgbm_real_test_mae = fit_lightgbm_full_and_test(lightgbm_best_real, train_real, val_real, test)

print(f"LightGBM all: validation MAE = {lightgbm_all_val_mae:.2f} сек.; test MAE = {lightgbm_all_test_mae:.2f} сек.")
print(f"LightGBM real: validation MAE = {lightgbm_real_val_mae:.2f} сек.; test MAE = {lightgbm_real_test_mae:.2f} сек.")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The ar

LightGBM all: validation MAE = 90.85 сек.; test MAE = 78.11 сек.
LightGBM real: validation MAE = 91.20 сек.; test MAE = 77.24 сек.


In [68]:
print(f"Baseline MAE: {base_mae:.2f} секунд")
print(f"MAE на ElasticNet: {linreg_real_test_mae:.2f} сек.")
print(f"MAE на CatBoost: {catboost_all_test_mae:.2f} сек.")
print(f"MAE на XGBoost: {xgboost_real_test_mae:.2f} сек.")
print(f"MAE на LightGBM: {lightgbm_real_test_mae:.2f} сек.")

Baseline MAE: 93.36 секунд
MAE на ElasticNet: 92.91 сек.
MAE на CatBoost: 48.84 сек.
MAE на XGBoost: 73.42 сек.
MAE на LightGBM: 77.24 сек.


In [69]:
Path("models").mkdir(exist_ok=True)
catboost_model_all.save_model("models/catboost_all.cbm")